# AMML Rank Gated

CPU-воспроизведение сабмита 58606250. Установите requirements.txt и скачайте оригинальный архив соревнования, как описано в README.

Ноутбук заново обучает финальный ранкер на входных данных Kaggle и сохранённых промежуточных оценках моделей. Следующая ячейка содержит сжатый архив числовых признаков.


In [ ]:
CACHE_BASE64 = (
    "..."
)


## Подготовка данных

По умолчанию ноутбук ищет архив или три исходных CSV в рабочем каталоге. Предсказания и отчёты сохраняются в outputs/.


In [ ]:
import base64
import io
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

data_path = Path(os.environ.get("AMML_DATA", ".")).expanduser().resolve()
raw_names = ("train.csv", "test.csv", "sample_submission.csv")
if data_path.is_dir() and not all((data_path / name).is_file() for name in raw_names):
    data_path /= "amml-ranking-competition-2026.zip"

if not data_path.exists():
    raise FileNotFoundError(
        "Скачайте архив соревнования с Kaggle после входа и положите его рядом с ноутбуком "
        "или задайте AMML_DATA как путь к архиву или каталогу с тремя исходными CSV."
    )

feature_dir = TemporaryDirectory(prefix="amml-features-")
root = Path(feature_dir.name)
with ZipFile(io.BytesIO(base64.b64decode(CACHE_BASE64))) as cache:
    for name in cache.namelist():
        (root / name).write_bytes(cache.read(name))
del CACHE_BASE64

if data_path.is_dir():
    data = data_path
else:
    data = root
    with ZipFile(data_path) as inputs:
        for name in raw_names:
            (data / name).write_bytes(inputs.read(name))

output_dir = Path(os.environ.get("AMML_OUTPUT", "outputs")).expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)


## Очистка данных и разбиение по запросам


In [ ]:
import pandas as pd
import numpy as np
import html
import re
import unicodedata


TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
SPACES = re.compile(r"\s+")
INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
MISSING = {"", "null", "none", "nan", "-", "n/a"}


def clean_text(value):
    value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
    value = TAG.sub(" ", value.translate(INVISIBLE))
    value = SPACES.sub(" ", value).strip()
    return "" if value.lower() in MISSING else value

def load_split(split):
    frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
    frame["query"] = frame["query"].map(clean_text)
    frame["text_clean"] = frame["text"].map(clean_text)
    frame["title_clean"] = frame["title"].map(clean_text)
    return frame

train, test = load_split("train"), load_split("test")
"""Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

import re


def _continuation_characters():
    characters = []
    for byte in range(0x80, 0xC0):
        try:
            characters.append(bytes([byte]).decode("cp1251"))
        except UnicodeDecodeError:
            continue
    return re.escape("".join(characters))


_PAIR = rf"[РС][{_continuation_characters()}]"
_RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
_MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
_LATIN = re.compile(r"[A-Za-z]")
_CYRILLIC = re.compile(r"[А-Яа-яЁё]")
_LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


def repair_encoding(value):
    """Reverse a validated whole-passage encoding error or recover individual runs."""
    if _RUN.search(value) is None:
        return value
    try:
        return value.encode("cp1251").decode("utf-8")
    except UnicodeError:
        pass

    def decode(match):
        original = match.group()
        candidate = original.encode("cp1251").decode("utf-8")
        return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

    return _RUN.sub(decode, value)


def repair_lookalikes(value):
    """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
    def restore(match):
        word = match.group()
        latin = _LATIN.findall(word)
        if not latin or len(_CYRILLIC.findall(word)) < 2:
            return word
        if all(ord(letter) in _LOOKALIKES for letter in latin):
            return word.translate(_LOOKALIKES)
        return word

    return _MIXED_WORD.sub(restore, value)


## Признаки и оценка NDCG@10


In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupShuffleSplit

def cached(name):
    with np.load(next(root.rglob(name)), allow_pickle=False) as source:
        result = pd.DataFrame({key: source[key] for key in source.files if key != "ids"}, index=source["ids"])
    assert result.index.is_unique and np.isfinite(result.to_numpy()).all()
    return result


def query_split(seed):
    _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
        train, groups=train["query_id"]
    ))
    return set(train.iloc[rows]["query_id"])


folds = {seed: query_split(seed) for seed in (41, 73)}
fit_queries = set.union(*folds.values())
remainder = set(train["query_id"]) - fit_queries
old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
audit = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
fresh = pd.read_csv(next(root.rglob("fresh_qwen8_scores.csv")))
fresh_queries = set(fresh["query_id"])
assert len(fresh_queries) == 350 and fresh_queries.isdisjoint(fit_queries | audit | old_audit)
validation = train[train["query_id"].isin(fit_queries | audit | fresh_queries)]
frame = pd.concat([validation, test], ignore_index=True)

TOKEN = re.compile(r"[^\W_]+")
SENTENCE = re.compile(r"(?<=[.!?])\s+")

def tokens(text):
    return TOKEN.findall(text.casefold().replace("ё", "е"))

def query_window(query, text, limit):
    if len(text) <= limit:
        return text
    terms = {word[:5] for word in tokens(query) if len(word) >= 3}
    if not terms:
        return text[:limit]
    sentences = SENTENCE.split(text)
    offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
    candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
    candidates.append(len(text) - limit)
    windows = [text[start:start + limit] for start in candidates]
    return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


original_graph = pd.read_csv(next(root.rglob("full_graph_scores.csv")))
graph = pd.concat([original_graph, fresh], ignore_index=True).set_index("id", verify_integrity=True)
base = frame["id"].map(graph["base"])
win = frame["id"].map(graph["window"])
full_score = frame["id"].map(graph["full"])
assert np.isfinite(pd.concat([base, win, full_score], axis=1).to_numpy()).all()
full = frame.copy()
full["title_clean"] = full["title_clean"].map(repair_lookalikes)
full["text_clean"] = [
    query_window(query, clean_text(repair_lookalikes(repair_encoding(text))), 1800)
    for query, text in zip(full["query"], full["text"])
]

def group_z(values):
    group = values.groupby(frame["query_id"])
    return (values - group.transform("mean")) / group.transform("std").fillna(0).clip(lower=1e-6)

semantic = pd.concat([cached(filename) for filename in (
    "bge_scores.npz", "horizon_scores.npz", "horizon_base_scores.npz", "lamar_scores.npz"
)], axis=1)
assert semantic.index.is_unique
features = pd.DataFrame({"base": base, "window": win, "full": full_score, "title_empty": frame["title_clean"].eq("")})
for name in semantic:
    feature = frame["id"].map(semantic[name])
    features[name] = group_z(feature)
    features[f"{name}_raw"] = feature
for name, value in (("full_minus_window", full_score - win), ("full_minus_base", full_score - base)):
    features[name] = value
features["words"] = full["text_clean"].map(lambda text: len(tokens(text)))
features["title_overlap"] = [
    len(set(tokens(q)).intersection(tokens(t))) for q, t in zip(full["query"], full["title_clean"])
]

def predict(query_ids):
    training = frame["query_id"].isin(query_ids)
    model = CatBoostClassifier(iterations=240, depth=3, learning_rate=.04, l2_leaf_reg=6,
                               random_seed=42, verbose=False, thread_count=4, allow_writing_files=False)
    model.fit(features.loc[training], frame.loc[training, "relevance"].astype(int))
    stack = group_z(pd.Series(model.predict_proba(features)[:, 1], index=frame.index))
    return pd.DataFrame({
        "original": .30 * base + .20 * win + .22 * full_score + .28 * stack,
        "full_lean": .24 * base + .15 * win + .33 * full_score + .28 * stack,
        "full_half": .18 * base + .18 * win + .36 * full_score + .28 * stack,
    })

def evaluate(predictions, queries, name):
    records = []
    for query_id, group in frame[frame["query_id"].isin(queries)].groupby("query_id"):
        y = group["relevance"].astype(int).to_numpy()
        metrics = {
            c: float(ndcg_score([y], [predictions.loc[group.index, c]], k=10)) if len(y) > 1 else float(y[0] > 0)
            for c in predictions
        }
        records.append({"query_id": query_id, "source": group["ingest_source"].mode().iat[0], **metrics})
    report = pd.DataFrame(records)
    print(name, report[list(predictions)].mean().round(6).to_dict(), flush=True)
    print(report.groupby("source")[list(predictions)].mean().round(6), flush=True)
    report.to_csv(output_dir / f"{name}.csv", index=False)
    return report


## Парное дообучение и поправка для неоднозначных запросов


In [ ]:
from catboost import CatBoostRanker, Pool

labeled = frame["relevance"].notna()
old_qwen = pd.read_csv(next(root.rglob("qwen8_validation_scores.csv"))).set_index("id", verify_integrity=True)["qwen8"]
all_qwen = pd.concat([old_qwen, fresh.set_index("id")["qwen8"]], verify_integrity=True)
qwen = frame["id"].map(all_qwen)
previous = pd.read_csv(next(root.rglob("qwen8_submission.csv"))).set_index("id", verify_integrity=True)["prediction"]
old_jina = pd.read_csv(next(root.rglob("jina_full_half.csv"))).set_index("id", verify_integrity=True)["prediction"]
deployed = frame["id"].map(previous)
qwen_test = (deployed - .8 * group_z(frame["id"].map(old_jina))) / .2
qwen = qwen.combine_first(qwen_test)
assert np.isfinite(qwen).all()
matrix = features.assign(
    qwen=qwen,
    qwen_rank=qwen.groupby(frame["query_id"]).rank(pct=True),
    qwen_gap=qwen.groupby(frame["query_id"]).transform("max") - qwen,
    jina_qwen_agreement=qwen - .25 * base - .25 * win - .50 * full_score,
    jina_views_spread=features[["base", "window", "full"]].std(axis=1),
)

public_best = pd.read_csv(next(root.rglob("robust_rank_submission.csv"))).set_index("id", verify_integrity=True)["prediction"]
public_best = frame["id"].map(public_best)
frozen = group_z(.8 * group_z(.25 * base + .25 * win + .50 * full_score) + .2 * qwen)
settings = (.15, .30, .50)

def focused_pool(indices):
    sampled = frame.loc[indices, ["query_id", "relevance"]].reset_index(drop=True)
    strength = frozen.loc[indices].to_numpy()
    labels = sampled["relevance"].to_numpy()
    pairs, pair_weights = [], []
    for group in sampled.groupby("query_id", sort=False).indices.values():
        relevant, irrelevant = group[labels[group] == 1], group[labels[group] == 0]
        if not len(relevant) or not len(irrelevant):
            continue
        actual = group[np.argsort(-strength[group], kind="stable")]
        position = {index: rank for rank, index in enumerate(actual)}
        discount = {index: 1 / np.log2(rank + 2) if rank < 10 else 0 for index, rank in position.items()}
        positive, negative = np.meshgrid(relevant, irrelevant, indexing="ij")
        positive, negative = positive.ravel(), negative.ravel()
        gain = np.array([abs(discount[a] - discount[b]) for a, b in zip(positive, negative)])
        pair_weight = .05 + gain * (.25 + .75 * np.exp(-np.abs(strength[positive] - strength[negative])))
        pair_weights.extend(pair_weight / pair_weight.sum())
        pairs.extend(zip(positive.tolist(), negative.tolist()))
    return Pool(matrix.loc[indices], label=sampled["relevance"].astype(int),
                group_id=sampled["query_id"], pairs=pairs, pairs_weight=pair_weights,
                baseline=2 * strength)

def run_model(queries):
    ordered = frame.loc[frame["query_id"].isin(queries)].sort_values(["query_id", "id"]).index
    groups = frame.loc[ordered, "query_id"]
    labels = frame.loc[ordered, "relevance"].astype(int)
    champion_model = CatBoostRanker(iterations=320, depth=3, learning_rate=.035, l2_leaf_reg=8,
                                    loss_function="PairLogit", random_seed=42, thread_count=4,
                                    verbose=False, allow_writing_files=False)
    champion_model.fit(Pool(matrix.loc[ordered], label=labels, group_id=groups))
    rank = group_z(pd.Series(champion_model.predict(matrix), index=frame.index))
    deployed_val = deployed.combine_first(.8 * group_z(predict(queries)["full_half"]) + .2 * qwen)
    champion = group_z(public_best.combine_first(.65 * group_z(deployed_val) + .35 * rank))

    extended = frame.loc[frame["query_id"].isin(queries | audit)].sort_values(["query_id","id"]).index
    residual = CatBoostRanker(iterations=360, depth=3, learning_rate=.035, l2_leaf_reg=8,
                               loss_function="PairLogit", random_seed=42, thread_count=4,
                               verbose=False, allow_writing_files=False)
    residual.fit(focused_pool(extended))
    corrections = pd.Series(residual.predict(matrix), index=frame.index)
    champion_order = champion.groupby(frame["query_id"]).rank(ascending=False, method="first")
    within_top3 = champion_order.le(3)
    leading = champion.where(champion_order.eq(1)).groupby(frame["query_id"]).transform("max")
    next_best = champion.where(champion_order.eq(2)).groupby(frame["query_id"]).transform("max")
    ambiguity = (1 - (leading - next_best).fillna(2) / 1.25).clip(lower=0, upper=1)
    light = champion + .15 * ambiguity * corrections
    medium = champion + .30 * ambiguity * corrections
    # Preserve all ranks below the shortlist and retain the score scale.
    reordered = champion.copy()
    for _, rows in frame.loc[within_top3].groupby("query_id"):
        original = champion.loc[rows.index].sort_values(ascending=False, kind="stable")
        refined = medium.loc[rows.index].sort_values(ascending=False, kind="stable")
        reordered.loc[refined.index] = original.to_numpy()
    return pd.DataFrame({
        "champion": champion,
        "gated_light": light,
        "gated_medium": medium,
        "gated_top3": reordered,
    })


## Валидация и формирование сабмита


In [ ]:
reports = [
    evaluate(run_model(fit_queries - queries), queries, f"gated_seed_{seed}")
    for seed, queries in folds.items()
]
metrics = pd.DataFrame([r.mean(numeric_only=True) for r in reports])
deltas = metrics.drop(columns="champion").subtract(metrics["champion"], axis=0)
dev_choice = deltas.min().idxmax()
print("Best dev candidate:", dev_choice)
# Freeze the method selected for the original Kaggle submission.
chosen = "gated_medium"
pred = run_model(fit_queries)
report = evaluate(pred[["champion", chosen]], fresh_queries, "gated_fresh_audit")
articles = set(frame.loc[frame["query_id"].isin(fit_queries | audit), "article_id"])
novel = {q for q, rows in frame[frame["query_id"].isin(fresh_queries)].groupby("query_id")
         if set(rows["article_id"]).isdisjoint(articles)}
gains = report[chosen] - report["champion"]
novel_gain = gains[report["query_id"].isin(novel)].mean()
print("Selected", chosen, "worst fold", deltas[chosen].min(), "fresh", gains.mean(),
      "novel count and delta", len(novel), novel_gain, flush=True)
sample = pd.read_csv(data / "sample_submission.csv")
values = sample["id"].map(pd.Series(pred[chosen].to_numpy(), index=frame["id"]))
assert np.isfinite(values).all()
sample.assign(prediction=values).to_csv(output_dir / "rank_gated_submission.csv", index=False)
